# ⚡ XGBoost + LightGBM Stacking Ensemble — Financial Time Series

**Chạy được cả LOCAL lẫn Google Colab.**

| # | Kỹ thuật | Mô tả |
|---|----------|-------|
| 1 | **Auto Lag-Feature Engineering** | Tự động tạo lag, rolling mean/std/min/max, EWMA |
| 2 | **Optuna + MedianPruner** | Bayesian HPO, dừng sớm trial kém hơn median |
| 3 | **OOF Stacking Ensemble** | XGB + LGB base → Ridge meta-learner (TimeSeriesSplit) |
| 4 | **Conformal Prediction Intervals** | Non-parametric PI với coverage guarantee |

## 📦 Section 0 — Cài đặt dependencies

In [10]:
# Chỉ cần chạy 1 lần
import subprocess, sys
pkgs = ['xgboost>=2.0.0', 'lightgbm>=4.0.0', 'optuna>=3.0.0',
        'scikit-learn>=1.3.0', 'plotly>=5.0.0']

for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('Cai dat hoan tat')

Cai dat hoan tat


## 📁 Section 1 — Tự động nhận diện môi trường & import

In [11]:
import sys, os
from pathlib import Path

# ── Tự động nhận diện LOCAL vs COLAB ─────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Moi truong: {"Google Colab" if IS_COLAB else "Local"}')

if IS_COLAB:
    # ── COLAB: Mount Drive hoặc upload file ──────────────────────────────────
    # Option A: Mount Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # PROJECT_DIR = '/content/drive/MyDrive/Financial-Forecast'
    # DATA_DIR    = PROJECT_DIR + '/data'

    # Option B: Upload trực tiếp
    # from google.colab import files
    # files.upload()  # upload gbm_model.py, train.json, test.json

    PROJECT_DIR = '/content'
    DATA_DIR    = '/content'
else:
    # ── LOCAL: tự động tìm project root ──────────────────────────────────────
    # Notebook nằm trong notebooks/, project root là thư mục cha
    _nb_dir = Path(globals().get('__vsc_ipynb_file__',
                   Path.cwd() / 'notebooks' / 'gbm_ensemble_colab.ipynb')).parent
    PROJECT_DIR = str(_nb_dir.parent)          # project root
    DATA_DIR    = str(_nb_dir.parent / 'colab_data')  # colab_data/

print(f'PROJECT_DIR = {PROJECT_DIR}')
print(f'DATA_DIR    = {DATA_DIR}')

# Thêm project root vào sys.path để import models/
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Verify
assert Path(PROJECT_DIR, 'models', 'gbm_model.py').exists(), \
    f'Khong tim thay models/gbm_model.py tai {PROJECT_DIR}. Kiem tra PROJECT_DIR.'

Moi truong: Local
PROJECT_DIR = c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection
DATA_DIR    = c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data


In [12]:
from models.gbm_model import GBMConfig, GBMForecaster
print('Import GBMForecaster thanh cong')

Import GBMForecaster thanh cong


## ⚙️ Section 2 — Cấu hình

In [13]:
import os

TRAIN_JSON = os.path.join(DATA_DIR, 'train.json')
TEST_JSON  = os.path.join(DATA_DIR, 'test.json')

# Kiểm tra file tồn tại
from pathlib import Path
assert Path(TRAIN_JSON).exists(), (
    f'Khong tim thay {TRAIN_JSON}.\n'
    'LOCAL: Chay truoc: python scripts/export_for_colab.py\n'
    'COLAB: Upload train.json len /content/ hoac Drive'
)
print(f'train.json: {TRAIN_JSON} ✓')
print(f'test.json : {TEST_JSON} ✓')

OUTPUT_DIR = os.path.join(PROJECT_DIR, 'gbm_output')

cfg = GBMConfig(
    target='close',
    group_col='symbol',
    time_col='open_time',
    lags=[1, 2, 3, 5, 7, 14, 21],
    rolling_windows=[7, 14, 30],
    ewma_spans=[7, 14],
    max_prediction_length=7,
    n_optuna_trials=20,
    optuna_cv_folds=3,
    n_cv_folds=5,
    meta_learner='ridge',
    conformal_alpha=0.10,
    calibration_ratio=0.15,
    output_dir=OUTPUT_DIR,
)

print(f'\nConfig OK | output -> {OUTPUT_DIR}')

train.json: c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\train.json ✓
test.json : c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\test.json ✓

Config OK | output -> c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\gbm_output


## 📂 Section 3 — Load data

In [14]:
import pandas as pd
import numpy as np

forecaster = GBMForecaster(cfg)

df_train_raw, df_cal_raw, df_test_raw = forecaster.load_and_split(
    train_path=TRAIN_JSON,
    test_path=TEST_JSON,
    symbol='BTC/USDT',
)

print(f'df_train_raw : {df_train_raw.shape}')
print(f'df_cal_raw   : {df_cal_raw.shape}')
print(f'df_test_raw  : {df_test_raw.shape}')
print(f'Columns: {list(df_train_raw.columns)[:10]} ...')

10:00:05 [INFO] GBMForecaster ready | target=close | pred_len=7 | lags=[1, 2, 3, 5, 7, 14, 21] | rolling=[7, 14, 30]
10:00:05 [INFO] Loading train: c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\train.json
10:00:05 [INFO] Loading test:  c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\colab_data\test.json
10:00:05 [INFO] Split | train=512 | calibration=91 | test=107


df_train_raw : (512, 47)
df_cal_raw   : (91, 47)
df_test_raw  : (107, 47)
Columns: ['open_time', 'open', 'high', 'low', 'close', 'volume', 'return_pct', 'log_return', 'ma_7', 'ma_25'] ...


## 🔧 Section 4 — [Kỹ thuật 1] Auto Lag-Feature Engineering

In [15]:
df_train = forecaster.build_lag_features(df_train_raw, drop_na=True)
df_cal   = forecaster.build_lag_features(df_cal_raw,   drop_na=True)
df_test  = forecaster.build_lag_features(df_test_raw,  drop_na=True)

lag_cols = [c for c in df_train.columns if f"{cfg.target}_" in c]
print(f'Lag/Rolling features ({len(lag_cols)}): {lag_cols}')
print(f'Total features: {df_train.shape[1]}')

10:00:05 [INFO] Lag features built: 22 new columns | rows: 512 → 491
10:00:05 [INFO] Lag features built: 22 new columns | rows: 91 → 70
10:00:05 [INFO] Lag features built: 22 new columns | rows: 107 → 86


Lag/Rolling features (22): ['close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_5', 'close_lag_7', 'close_lag_14', 'close_lag_21', 'close_roll_mean_7', 'close_roll_std_7', 'close_roll_min_7', 'close_roll_max_7', 'close_roll_mean_14', 'close_roll_std_14', 'close_roll_min_14', 'close_roll_max_14', 'close_roll_mean_30', 'close_roll_std_30', 'close_ewma_7', 'close_ewma_14', 'close_diff_1', 'close_diff_7', 'close_mom_7']
Total features: 69


## 🔍 Section 5 — [Kỹ thuật 2] Optuna + MedianPruner HPO

In [16]:
RUN_HPO = True   # Dat False de skip

if RUN_HPO:
    hpo_results = forecaster.tune_all(df_train, n_trials=cfg.n_optuna_trials)
    print('\nBest XGB params:', hpo_results['xgb'])
    print('Best LGB params:', hpo_results['lgb'])
else:
    print('Skip HPO')

10:00:05 [INFO] Tuning XGBoost (20 trials) ...
Best trial: 1. Best value: 5378.65: 100%|██████████| 20/20 [00:43<00:00,  2.20s/it]
10:00:49 [INFO] XGB best params: {'n_estimators': 1486, 'max_depth': 3, 'learning_rate': 0.07075463042863876, 'subsample': 0.5485521832646199, 'colsample_bytree': 0.9311710377951222, 'reg_alpha': 7.109678963000493e-08, 'reg_lambda': 0.0035859136004253536, 'min_child_weight': 2} | val_RMSE=5378.6477
10:00:49 [INFO] Tuning LightGBM (20 trials) ...
Best trial: 16. Best value: 5584.38: 100%|██████████| 20/20 [00:06<00:00,  3.11it/s]
10:00:55 [INFO] LGB best params: {'n_estimators': 826, 'num_leaves': 66, 'learning_rate': 0.03559213928543985, 'subsample': 0.7468564635612749, 'colsample_bytree': 0.6008112698406154, 'reg_alpha': 3.42451385818481e-08, 'reg_lambda': 2.78801657280332, 'min_child_samples': 39} | val_RMSE=5584.3756
10:00:55 [INFO] HPO results saved → c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\gbm_output\hpo_result


Best XGB params: {'n_estimators': 1486, 'max_depth': 3, 'learning_rate': 0.07075463042863876, 'subsample': 0.5485521832646199, 'colsample_bytree': 0.9311710377951222, 'reg_alpha': 7.109678963000493e-08, 'reg_lambda': 0.0035859136004253536, 'min_child_weight': 2}
Best LGB params: {'n_estimators': 826, 'num_leaves': 66, 'learning_rate': 0.03559213928543985, 'subsample': 0.7468564635612749, 'colsample_bytree': 0.6008112698406154, 'reg_alpha': 3.42451385818481e-08, 'reg_lambda': 2.78801657280332, 'min_child_samples': 39}


## 🏗️ Section 6 — [Kỹ thuật 3] OOF Stacking Ensemble

In [17]:
forecaster.fit_stacking(df_train=df_train, df_cal=df_cal)

print('Stacking fitted!')
for r in forecaster.oof_scores_:
    print(f"  Fold {r['fold']}: XGB={r['xgb_rmse']:.0f} | LGB={r['lgb_rmse']:.0f}")

10:00:56 [INFO] Generating OOF predictions (5 folds)...
10:00:56 [INFO]   Fold 1/5 | XGB_RMSE=4590.1156 | LGB_RMSE=3848.8611
10:00:56 [INFO]   Fold 2/5 | XGB_RMSE=5646.8032 | LGB_RMSE=7818.5616
10:00:56 [INFO]   Fold 3/5 | XGB_RMSE=7878.5101 | LGB_RMSE=8736.9992
10:00:56 [INFO]   Fold 4/5 | XGB_RMSE=2926.4144 | LGB_RMSE=4107.8179
10:00:57 [INFO]   Fold 5/5 | XGB_RMSE=2202.3832 | LGB_RMSE=2400.9310
10:00:57 [INFO] Meta-learner (ridge) fitted on OOF matrix shape (405, 2)
10:00:58 [INFO] XGBoost retrained on full data.
10:00:58 [INFO] LightGBM retrained on full data.
10:00:58 [INFO] Feature importance saved → c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\gbm_output\feature_importance.json


Stacking fitted!
  Fold 1: XGB=4590 | LGB=3849
  Fold 2: XGB=5647 | LGB=7819
  Fold 3: XGB=7879 | LGB=8737
  Fold 4: XGB=2926 | LGB=4108
  Fold 5: XGB=2202 | LGB=2401


## 🎯 Section 7 — [Kỹ thuật 4] Conformal Prediction Calibration

In [18]:
forecaster.calibrate_conformal(df_cal)
print(f'q90 (+/- interval for 90% coverage): {forecaster._q90:.2f}')

10:00:58 [INFO] Conformal calibrated on 69 samples | q80=1606.1865 | q90=1713.9597 | q95=1730.4379


q90 (+/- interval for 90% coverage): 1713.96


## 🔮 Section 8 — Prediction & Evaluation

In [19]:
# Batch prediction cho toàn bộ test set
predictions_batch = forecaster.predict_batch(df_test)

# Ground truth
X_te, y_true, _ = forecaster._make_xy(df_test, horizon=0)

# Metrics
metrics = forecaster.evaluate(y_true, predictions_batch)

print(f"MAE    : {metrics['MAE']:.2f}")
print(f"RMSE   : {metrics['RMSE']:.2f}")
print(f"MAPE%  : {metrics['MAPE_%']:.2f}%")
if 'Coverage_90_%' in metrics:
    print(f"Cov90% : {metrics['Coverage_90_%']:.1f}%  (target >= 90%)")

10:00:59 [INFO] ── Evaluation Metrics ──────────────────────
10:00:59 [INFO]   MAE                      : 1385.3675
10:00:59 [INFO]   RMSE                     : 1776.9767
10:00:59 [INFO]   MAPE_%                   : 2.1727
10:00:59 [INFO]   sMAPE_%                  : 2.1368
10:00:59 [INFO]   Winkler_90               : 9249.5074
10:00:59 [INFO]   Coverage_90_%            : 68.6047
10:00:59 [INFO]   Winkler_80               : 6481.6835
10:00:59 [INFO]   Coverage_80_%            : 63.9535
10:00:59 [INFO]   OOF_XGB_RMSE_mean        : 4648.8453
10:00:59 [INFO]   OOF_LGB_RMSE_mean        : 5382.6342
10:00:59 [INFO] ────────────────────────────────────────────


MAE    : 1385.37
RMSE   : 1776.98
MAPE%  : 2.17%
Cov90% : 68.6%  (target >= 90%)


In [20]:
# Recursive multi-step: 7 bước phía trước
recursive_pred = forecaster.recursive_predict(
    df_test=df_test,
    steps=cfg.max_prediction_length,
    history_df=df_train.tail(30),
)
print('\nRecursive 7-step forecast:')
print(recursive_pred[['step', 'y_pred', 'lower_90', 'upper_90']].to_string(index=False))

10:00:59 [INFO] Lag features built: 22 new columns | rows: 116 → 116
10:00:59 [INFO] Recursive predict complete | steps=7



Recursive 7-step forecast:
 step       y_pred     lower_90     upper_90
    1 66077.699853 64363.740133 67791.659573
    2 66077.699853 64363.740133 67791.659573
    3 66077.699853 64363.740133 67791.659573
    4 66077.699853 64363.740133 67791.659573
    5 66077.699853 64363.740133 67791.659573
    6 66077.699853 64363.740133 67791.659573
    7 66077.699853 64363.740133 67791.659573


## 💾 Section 9 — Lưu kết quả JSON

In [21]:
import json

RESULT_PATH = os.path.join(OUTPUT_DIR, 'gbm_results.json')

forecaster.save_results(
    output_path=RESULT_PATH,
    y_true=y_true,
    predictions=predictions_batch,
    extra={
        'symbol': 'BTC/USDT',
        'recursive_forecast': recursive_pred.to_dict(orient='records'),
    }
)

print(f'Results saved -> {RESULT_PATH}')

# Preview
with open(RESULT_PATH) as f:
    r = json.load(f)
for k, v in r.items():
    if isinstance(v, dict): print(f'  {k}: dict({len(v)} keys)')
    elif isinstance(v, list): print(f'  {k}: list({len(v)} items)')
    else: print(f'  {k}: {str(v)[:60]}')

10:00:59 [INFO] ── Evaluation Metrics ──────────────────────
10:00:59 [INFO]   MAE                      : 1385.3675
10:00:59 [INFO]   RMSE                     : 1776.9767
10:00:59 [INFO]   MAPE_%                   : 2.1727
10:00:59 [INFO]   sMAPE_%                  : 2.1368
10:00:59 [INFO]   Winkler_90               : 9249.5074
10:00:59 [INFO]   Coverage_90_%            : 68.6047
10:00:59 [INFO]   Winkler_80               : 6481.6835
10:00:59 [INFO]   Coverage_80_%            : 63.9535
10:00:59 [INFO]   OOF_XGB_RMSE_mean        : 4648.8453
10:00:59 [INFO]   OOF_LGB_RMSE_mean        : 5382.6342
10:00:59 [INFO] ────────────────────────────────────────────
10:00:59 [INFO] Results saved → c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\gbm_output\gbm_results.json


Results saved -> c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\gbm_output\gbm_results.json
  config: dict(17 keys)
  best_hparams: dict(2 keys)
  oof_scores: list(5 items)
  conformal: dict(4 keys)
  metrics: dict(10 keys)
  feature_importance: dict(2 keys)
  predictions: dict(5 keys)
  y_true: list(86 items)
  symbol: BTC/USDT
  recursive_forecast: list(7 items)


## 📈 Section 10 — Visualization

In [22]:
import plotly.graph_objects as go

n_plot = min(len(y_true), len(predictions_batch))
x_axis = list(range(n_plot))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_axis + x_axis[::-1],
    y=predictions_batch['lower_90'][:n_plot].tolist() +
      predictions_batch['upper_90'][:n_plot].tolist()[::-1],
    fill='toself', fillcolor='rgba(255,149,0,0.12)',
    line=dict(color='rgba(0,0,0,0)'), name='90% Conformal PI',
))

fig.add_trace(go.Scatter(
    x=x_axis, y=y_true[:n_plot],
    name='Actual', line=dict(color='#c9d1d9', width=1.5),
))

fig.add_trace(go.Scatter(
    x=x_axis, y=predictions_batch['y_pred'][:n_plot].values,
    name='Stacking Ensemble', line=dict(color='#ff9500', width=2),
))

fig.update_layout(
    title=f"XGB+LGB Stacking | {cfg.target.upper()} | MAE={metrics['MAE']:.0f} | MAPE={metrics['MAPE_%']:.2f}%",
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#8b949e'), hovermode='x unified', height=480,
    xaxis=dict(title='Test Step', gridcolor='#21262d'),
    yaxis=dict(title=cfg.target.capitalize(), gridcolor='#21262d'),
    legend=dict(orientation='h', y=1.02, x=1, xanchor='right'),
)
fig.show()

plot_path = os.path.join(OUTPUT_DIR, 'forecast_plot.html')
fig.write_html(plot_path)
print(f'Plot saved -> {plot_path}')

Plot saved -> c:\Users\Dang_Thanh\Desktop\Financial Time Series Forecasting & Anomaly Detection\gbm_output\forecast_plot.html


---
## ✅ Xong!
Kết quả tại `gbm_output/gbm_results.json`